# CRT-SOHO train-only CIFAR-100 gates

This notebook reuses the frozen ViT feature cache, constructs a fixed sparse/WTA anchor once, and runs only the predeclared **training-validation** falsification gates. It never opens `test.pt` during selection and does not report held-out CIFAR-100 accuracy. Even when every gate passes, stop and review `gate_results.json` before any test run.

In [ ]:
# === Edit this cell only ===
REPO_GIT_URL = 'https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH = 'feature/crt-soho'  # push this branch before Colab
CHECKPOINT_SOURCE = 'huggingface'  # 'huggingface' or 'google_drive'
DRIVE_CHECKPOINT_PATH = '/content/drive/MyDrive/T-SOHO/model.safetensors'
WORK_DIR = '/content/SOHO-CL'
FEATURE_CACHE_DIR = '/content/tsoho_cifar100_cache'
CRT_GATE_CACHE_DIR = '/content/crt_soho_gate_cache'
OUTPUT_DIR = '/content/crt_soho_gate_outputs'
SEED = 1993
NUM_TASKS = 10
VALIDATION_FRACTION = 0.10
BATCH_SIZE = 128
ANCHOR_BATCH_SIZE = 1024
ANCHOR_DIM = 1024  # pilot size; do not change after seeing test results
SYNAPTIC_DEGREE = 300
CODING_LEVEL = 0.30
STATISTICS_DTYPE = 'float32'  # practical T4 pilot; solver residual is logged
ANCHOR_RIDGES = '0.01,0.1,1.0'
RESIDUAL_RIDGES = '0.1,1.0'
COMPLEMENT_RIDGES = '0.1,1.0'
RANKS = '32,64,128'
TEMPERATURES = '0.5,1.0'
MINIMUM_FULL_GAIN = 0.10  # percentage points
MAXIMUM_LOW_RANK_GAP = 0.50  # percentage points
MINIMUM_CONFUSION_GAIN = 0.10  # percentage points
MAXIMUM_RELATIVE_SOLVER_RESIDUAL = 1e-4
CHECKPOINT_SIZE = 346284714
CHECKPOINT_SHA256 = '32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'


In [ ]:
# Fresh clone and environment.
import os, shutil, subprocess, sys, torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU.'
%cd /content
shutil.rmtree(WORK_DIR, ignore_errors=True)
subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, REPO_GIT_URL, WORK_DIR], check=True)
%cd {WORK_DIR}
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-kaggle.txt', 'kagglehub', 'huggingface_hub'], check=True)
subprocess.run(['nvidia-smi'], check=True)
subprocess.run(['git', 'log', '-1', '--oneline'], check=True)


In [ ]:
# Resolve the verified ViT checkpoint and public CIFAR-100 archive.
from pathlib import Path
if CHECKPOINT_SOURCE == 'google_drive':
    from google.colab import drive
    drive.mount('/content/drive')
    CHECKPOINT_PATH = DRIVE_CHECKPOINT_PATH
elif CHECKPOINT_SOURCE == 'huggingface':
    from huggingface_hub import hf_hub_download
    CHECKPOINT_PATH = hf_hub_download(repo_id='timm/vit_base_patch16_224.augreg2_in21k_ft_in1k', filename='model.safetensors')
else:
    raise ValueError("CHECKPOINT_SOURCE must be 'huggingface' or 'google_drive'")
import kagglehub
downloaded = Path(kagglehub.dataset_download('zaphat206/cifar-100'))
candidates = [downloaded, *downloaded.rglob('cifar-100')]
cifar_dir = next(p for p in candidates if (p / 'train').is_file() and (p / 'test').is_file() and (p / 'meta').is_file())
CIFAR_ROOT = str(cifar_dir)
assert Path(CHECKPOINT_PATH).is_file(), CHECKPOINT_PATH
print('checkpoint:', CHECKPOINT_PATH)
print('CIFAR-100:', CIFAR_ROOT)


In [ ]:
# Preflight and focused correctness tests.
subprocess.run([sys.executable, 'tools/checkpoint_preflight.py', '--root', CIFAR_ROOT, '--checkpoint', CHECKPOINT_PATH, '--checkpoint-size', str(CHECKPOINT_SIZE), '--checkpoint-sha256', CHECKPOINT_SHA256, '--seed', str(SEED), '--batch-size', str(BATCH_SIZE)], check=True)
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_crt_soho_math.py', 'tests/test_crt_gate_runner.py', 'tests/test_experiment_runner.py'], check=True)


In [ ]:
# Extract frozen ViT features only when the validated cache is absent.
# This is the slow image pass; runner output remains visible in real time.
if not Path(FEATURE_CACHE_DIR, 'metadata.json').is_file():
    command = [sys.executable, '-u', 'tools/experiment_runner.py', '--extract-features-only', '--root', CIFAR_ROOT, '--backbone-checkpoint', CHECKPOINT_PATH, '--backbone-checkpoint-size', str(CHECKPOINT_SIZE), '--backbone-checkpoint-sha256', CHECKPOINT_SHA256, '--feature-cache-dir', FEATURE_CACHE_DIR, '--output-dir', f'{OUTPUT_DIR}/feature_extract', '--dataset', 'CIFAR-100', '--model-name', 'vit_base_patch16_224', '--data-augmentation', 'vit', '--seed', str(SEED), '--num-classes', '100', '--num-tasks', str(NUM_TASKS), '--device', 'cuda', '--batch-size', str(BATCH_SIZE), '--num-workers', '2']
    print('Running:', ' '.join(command), flush=True)
    subprocess.run(command, check=True)
else:
    print('Using existing frozen-feature cache:', FEATURE_CACHE_DIR)


In [ ]:
# Prepare reusable TRAIN-only statistics and run staged gates.
# Progress is printed for every cache task and every analytic candidate.
command = [sys.executable, '-u', 'tools/crt_gate_runner.py', '--prepare-cache', '--run-gates', '--feature-cache-dir', FEATURE_CACHE_DIR, '--gate-cache-dir', CRT_GATE_CACHE_DIR, '--output-dir', OUTPUT_DIR, '--dataset', 'CIFAR-100', '--model-name', 'vit_base_patch16_224', '--num-classes', '100', '--num-tasks', str(NUM_TASKS), '--validation-fraction', str(VALIDATION_FRACTION), '--seed', str(SEED), '--device', 'cuda', '--anchor-dim', str(ANCHOR_DIM), '--synaptic-degree', str(SYNAPTIC_DEGREE), '--coding-level', str(CODING_LEVEL), '--statistics-dtype', STATISTICS_DTYPE, '--anchor-batch-size', str(ANCHOR_BATCH_SIZE), '--anchor-ridges', ANCHOR_RIDGES, '--residual-ridges', RESIDUAL_RIDGES, '--complement-ridges', COMPLEMENT_RIDGES, '--ranks', RANKS, '--temperatures', TEMPERATURES, '--minimum-full-gain', str(MINIMUM_FULL_GAIN), '--maximum-low-rank-gap', str(MAXIMUM_LOW_RANK_GAP), '--minimum-confusion-gain', str(MINIMUM_CONFUSION_GAIN), '--maximum-relative-solver-residual', str(MAXIMUM_RELATIVE_SOLVER_RESIDUAL)]
print('Running:', ' '.join(command), flush=True)
subprocess.run(command, check=True)


In [ ]:
# Inspect the decision. This cell deliberately does NOT launch test evaluation.
import json, pandas as pd
report = json.load(open(f'{OUTPUT_DIR}/gate_results.json'))
summary = pd.DataFrame([{**{'method': c['method']}, 'rank': c['rank'], 'anchor_ridge': c['anchor_ridge'], 'residual_ridge': c['residual_ridge'], 'complement_ridge': c['complement_ridge'], 'temperature': c['temperature'], 'validation_AA': c['validation_average_incremental_accuracy'], 'validation_final': c['validation_final_accuracy'], 'state_bytes': c['persistent_state_bytes'], 'solver_relative_residual_max': c['solver_relative_residual_max']} for c in report['candidates']])
display(summary.sort_values('validation_AA', ascending=False))
print(json.dumps({'status': report['status'], 'gates': report['gates'], 'held_out_test_authorized': report['held_out_test_authorized']}, indent=2))
if report['held_out_test_authorized']:
    print('All train-only gates passed. STOP HERE and send gate_results.json for review.')
else:
    print('A validation gate failed. Do not run held-out CIFAR-100 test.')


In [ ]:
# Download evidence. Gate cache is intentionally excluded because it can be large.
archive = '/content/crt_soho_gate_results.zip'
subprocess.run(['zip', '-r', archive, OUTPUT_DIR], check=True)
from google.colab import files
files.download(archive)
